# 1.3 Calculate saturation pressures and equilibrium fluid compositions from MI data using VolFe

<div style="background-color:#eef8fa; border-left:4px solid #24bdff; padding:10px; border-radius:4px;">
<b> 🧮 &nbsp; VolFe </b> is an open-source Python package for calculating melt-vapor equilibra in the CHOS+ system, including concentration, speciation, and isotope ratios.

More information on VolFe can be found at https://volfe.readthedocs.io/en/latest/

</div>

We can also calculate the pressure of vapor saturation and fluid composition using VolFe, which assumes the fluid and melt contain a variety of reduced and oxidised C-O-H-S-bearing species. This needs additional information, such as some estimate of an oxygen fugacity related variable (e.g., <i>f</i>O<sub>2</sub>, ΔFMQ, ΔNNO, Fe<sup>3+</sup>/Fe<sub>T</sub>, or S<sup>6+</sup>/S<sub>T</sub>). We'll use a value of ΔFMQ+0.2 for all MI, which is within the range described in Wieser et al. (2021), but in other calculations this can be specified for each composition individually as a column in the input spreadsheet.

## 1.3.1 Import packages and note versions

In [ ]:
# Packages that we will use in our code always get imported before we need them.
# This is canonically done at the top of a script.
# ⚠️ Note that this can take a few seconds if it's the first time you're importing these libraries.

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import VolFe as vf

import os
os.makedirs("output", exist_ok=True) # creates the output folder

When reporting calculations in manuscripts, it's important to know which version of the Python package the results you are showing used - so we can output those below. This notebook was created using VolFe: 1.0.3.

In [ ]:
print(f"\nVolFe: {vf.__version__}")

## 1.3.2 Import data

We'll import the melt inclusion compositions including the calculated temperatures from <a href="1_1_MI_Temperature_Thermobar.ipynb">1.1 Calculate temperature from MI data using Thermobar</a>

In [ ]:
# import melt inclusion dataset
df_MI = pd.read_csv("output/wieser2021_w_temperatures.csv")

# if you haven't run Exercise 1.1, you can grab the "answer key" file from here - remove the # at the start of the line below and then run the cell
#df_MI = pd.read_csv("answers/wieser2021_w_temperatures.csv")

## 1.3.3 Explore options

There are lots of different options available in VolFe. Similar to VESIcal, there are different H<sub>2</sub>O and CO<sub>2</sub> solubility models, but there are also different sulfide and sulfate solubility models and relationships between Fe<sup>3+</sup>/Fe<sub>T</sub> and <i>f</i>O<sub>2</sub> to choose from (as well as lots of parameters too!). 

Run the cell below to see all the parameters that can be changed in VolFe or head to the VolFe ReadTheDocs: https://volfe.readthedocs.io/en/latest/current_mdv.html

In [ ]:
help(vf.make_df_and_add_model_defaults)

Here we'll focus on the H<sub>2</sub>O, CO<sub>2</sub>, sulfide, and sulfate solubility options:

In [ ]:
# H2O solubility model
help(vf.C_H2O)

In [ ]:
# CO2 solubility model
help(vf.C_CO3)

In [ ]:
# sulfide solubility model
help(vf.C_S)

In [ ]:
# sulfate solubility model
help(vf.C_SO4)

We'll choose to use the water solubility model from Fig. S2 of Hughes et al. (2024); CO<sub>2</sub> solubility model from Dixon et al. (1995); sulfide solubility model from eq. (10.34) of O'Neill (2021); and sulfate solubility model from eq. (12a) of O'Neill & Mavrogenes (2022) - everything else will use the default options in VolFe. All options used are output in the results so you can check what has been used.

In [ ]:
# choose the options I want - everything else will use the default options
models_vf = [['water','Basalt_Hughes24'],['carbon dioxide','MORB_Dixon95'],['sulfide','ONeill21dil'],['sulfate','ONeill22dil']]

# turn to dataframe with correct column headers and indexes
models_vf = vf.make_df_and_add_model_defaults(models_vf)

## 1.3.4 Run calculations

VolFe also uses different column names... so we rename some of them...

In [ ]:
MI_vf=df_MI.copy().reset_index(drop=True)

# rename the columns we need for VolFe
MI_vf = MI_vf.rename(columns={
    'CO2_ppm': 'CO2ppm',
    'S': 'STppm',
    'Sample Name':'Sample',
})

And add ΔFMQ to each row...

In [ ]:
# Adds oxygen fugacity constraint to each row in the dataframe
MI_vf['DFMQ'] = 0.20

And then we run the calculations...

In [ ]:
# runs the calculation
results_pvsat_vf = vf.calc_Pvsat(MI_vf, models= models_vf)

# save the results
results_pvsat_vf.to_csv("output/results_pvsat_vf.csv")

# uncomment the line below to interrogate the resulting dataframe
#results_pvsat_vf

A warning was generated for these calculations:

``UserWarning: you entered more than one way to infer iron speciation, note that this calcualtion is only considering the entered DFMQ``

This is because the input DataFrame contains FeO, Fe<sub>2</sub>O<sub>3</sub>, and ΔFMQ, which means <i>f</i>O<sub>2</sub> is over constrained. VolFe has an order in which it will pick which <i>f</i>O<sub>2</sub> variable to use in the calculation - ΔFMQ is used in preference to FeO and Fe<sub>2</sub>O<sub>3</sub>, and the total FeO is calculated from FeO and Fe<sub>2</sub>O<sub>3</sub>.

## 1.3.5 Plotting

And we can plot the results!

In [ ]:
# if you haven't run this Exercise, you can grab the "answer key" files from here:
#results_pvsat_vf = pd.read_csv("answers/results_pvsat_vf.csv")

In [ ]:
fig, ((ax1, ax2, ax3),(ax4, ax5, ax6)) = plt.subplots(2, 3, figsize=(12,8)) # six panel figure with three columns and two rows

# volfe results in purple
df = results_pvsat_vf
ax1.plot(df['H2OT-eq_wtpc'], df['P_bar'], 'ok', mfc='purple')
ax2.plot(df['CO2T-eq_ppmw'], df['P_bar'], 'ok', mfc='purple')
ax3.plot(df['xgCO2_mf']*100, df['P_bar'], 'dk', mfc='purple')
ax4.plot(df['ST_ppmw'], df['P_bar'], 'ok', mfc='purple')
ax5.plot(df['Fe3+/FeT'], df['P_bar'], 'ok', mfc='purple')
ax6.plot(100*df['xgSO2_mf']/(df['xgCO2_mf']+df['xgSO2_mf']), df['P_bar'], 'dk', mfc='purple')

#axes labels
ax1.set_ylabel('P (bars)')
ax1.set_xlabel('H$_2$O [m] (wt%)')
ax2.set_xlabel('CO$_2$ [m] (ppm)')
ax3.set_xlabel('CO$_2$ [f] (mol%)')
ax4.set_ylabel('P (bars)')
ax4.set_xlabel('S [m] (ppm)')
ax5.set_xlabel('Fe$^{3+}$/Fe$_T$ [m]')
ax6.set_xlabel('SO$_2$/(CO$_2$+SO$_2$) [f] (mol%)')

plt.tight_layout()